In [1]:
from collections import Counter, defaultdict
from datetime import datetime
import json
import os
from pathlib import Path

# Configuration from your setup
STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 17

RECORDS_TO_PROCESS = 1000 + ID_SUM  # 1017
TARGET_STATUS_CLASS = str((SEED_DIGIT % 4) + 2)  # '2' -> 2xx

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

csv_path = DATA_DIR / "weblog.csv"
filtered_csv_path = OUTPUT_DIR / f"{STUDENT_ID}_weblog_filtered.csv"
json_summary_path = OUTPUT_DIR / f"{STUDENT_ID}_weblog_summary.json"
backup_json_path = BACKUP_DIR / f"{STUDENT_ID}_weblog_summary.bak"

total_processed = 0
target_class_count = 0
status_counts = Counter()
url_counts = Counter()
filtered_rows = []

# Define standard headers for reference and export
DEFAULT_HEADER = ["IP", "Time", "URL", "Status"]

try:
  with open(csv_path, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()
    
    # Check if the first line is a header (contains non-numeric status or alphabetic column names)
    # weblog.csv datasets often start directly with data rows like: 10.0.0.1,01/Jan/... ,/index.html,200
    first_parts = first_line.split(",")
    
    has_header = False
    if len(first_parts) >= 4:
      # If the 4th element is not a digit (e.g. it says 'status' or similar), it's a header
      if not first_parts[3].strip().isdigit():
        has_header = True

    # If it has a header, we skip it. If not, we process the first line as a data row.
    if not has_header:
      # Process the first line as data
      parts = first_parts
      if len(parts) >= 4:
        ip, timestamp, url, status = parts[0], parts[1], parts[2], parts[3]
        total_processed += 1
        status_code_prefix = status.strip()[0]
        status_counts[status.strip()] += 1
        if status_code_prefix == TARGET_STATUS_CLASS:
          target_class_count += 1
          url_counts[url] += 1
          filtered_rows.append(parts)

    # Process remaining lines line-by-line
    for line in f:
      if total_processed >= RECORDS_TO_PROCESS:
        break
      parts = [p.strip() for p in line.strip().split(",")]
      if len(parts) >= 4:
        ip, timestamp, url, status = parts[0], parts[1], parts[2], parts[3]
        total_processed += 1
        status_code_prefix = status.strip()[0]
        status_counts[status.strip()] += 1

        # Select records whose HTTP status belongs to target status class (2xx)
        if status_code_prefix == TARGET_STATUS_CLASS:
          target_class_count += 1
          url_counts[url] += 1
          filtered_rows.append(parts)

except FileNotFoundError:
  print(f"Error: {csv_path} not found. Please place it in the 'data/' directory.")

# Five most frequently requested URLs within target class
five_most_requested = url_counts.most_common(5)

# Export selected records to a filtered CSV file
with open(filtered_csv_path, "w", encoding="utf-8") as out_f:
  out_f.write(",".join(DEFAULT_HEADER) + "\n")
  for row in filtered_rows:
    out_f.write(",".join(row[:4]) + "\n")

# Write summary to JSON
summary_data = {
    "records_processed": total_processed,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": target_class_count,
    "status_counts": dict(status_counts),
    "five_most_requested_urls": five_most_requested,
}

with open(json_summary_path, "w", encoding="utf-8") as json_f:
  json.dump(summary_data, json_f, indent=4)

# Create a backup of the JSON file and verify that it exists
import shutil

shutil.copy(json_summary_path, backup_json_path)
backup_verified = backup_json_path.exists()
backup_size = backup_json_path.stat().st_size if backup_verified else 0

print("--- Exercise 1 Assessment Data (Header-Safe) ---")
print(f"Records processed: {total_processed}")
print(f"Target status class: {TARGET_STATUS_CLASS}xx")
print(f"Records in target class: {target_class_count}")
print(f"Count per status code: {dict(status_counts)}")
print(f"Five most requested URLs: {five_most_requested}")
print(f"Filtered CSV file: {filtered_csv_path}")
print(f"JSON summary file: {json_summary_path}")
print(f"Backup verification: Exists={backup_verified}, Size={backup_size} bytes")

Error: data\weblog.csv not found. Please place it in the 'data/' directory.
--- Exercise 1 Assessment Data (Header-Safe) ---
Records processed: 0
Target status class: 2xx
Records in target class: 0
Count per status code: {}
Five most requested URLs: []
Filtered CSV file: output\TUPM-26-4014_weblog_filtered.csv
JSON summary file: output\TUPM-26-4014_weblog_summary.json
Backup verification: Exists=True, Size=165 bytes
